# 中国5A景区检索示例

本示例演示如何使用 LlamaIndex + FAISS 向量存储构建中国5A景区的检索系统。

In [ ]:
# 导入必要的库
import os
# Note: all paths are cleared on purpose
os.environ["HF_HOME"] = "Your HF_HOME path"

from llama_index.core import VectorStoreIndex
from llama_index.core.readers import SimpleDirectoryReader
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.faiss import FaissVectorStore
import faiss

# 导入项目中的 RetrieverAdapter
import sys
sys.path.insert(0, "Your project path")
from aap_llamaindex.retriever import RetrieverAdapter
from aap_core.types import AgentMessage

In [ ]:
# 1. 加载数据目录中的所有 markdown 文件
data_dir = "Your data directory path"
reader = SimpleDirectoryReader(
    input_dir=data_dir,
    required_exts=[".md"],
)
documents = reader.load_data()

print(f"加载了 {len(documents)} 个文档")
print(f"文档列表: {[doc.metadata.get('file_name', 'unknown') for doc in documents[:10]]}")


加载了 198 个文档
文档列表: ['万佛湖.md', '万峰林.md', '三亚.md', '三峡大坝.md', '三河古镇.md', '三清山.md', '上海迪士尼乐园.md', '东方明珠广播电视塔.md', '东钱湖.md', '丹霞山.md']


In [4]:
# 2. 创建 FAISS 向量存储
# 使用 bge-base-zh-v1.5 中文嵌入模型
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-zh-v1.5")

# 创建 FAISS 向量存储 (维度与嵌入模型匹配，bge-base 输出 768 维)
d = 768
faiss_index = faiss.IndexFlatL2(d)
vector_store = FaissVectorStore(faiss_index=faiss_index)

# 3. 从文档创建向量索引
print("正在创建向量索引...")
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=embed_model,
    vector_store=vector_store,
)
print("向量索引创建完成！")


2026-05-17 09:28:47,032 - INFO - Load pretrained SentenceTransformer: BAAI/bge-base-zh-v1.5
2026-05-17 09:29:05,238 - INFO - 1 prompt is loaded, with the key: query


正在创建向量索引...
向量索引创建完成！


In [5]:
# 4. 配置检索器 - 返回最相关的 3 个结果
retriever = index.as_retriever(similarity_top_k=3)

# 5. 使用 RetrieverAdapter 包装检索器
adapter = RetrieverAdapter(retriever=retriever)
print("检索器配置完成！")


检索器配置完成！


In [6]:
# 6. 测试检索 - 查询示例
test_queries = [
    "中国最著名的宫殿是什么？",
    "有哪些著名的古城景区？",
    "中国有哪些著名的山脉？",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"查询: {query}")
    print(f"{'='*60}")

    message = AgentMessage(query=query)
    message = adapter(message)

    if message.context and "data" in message.context:
        for i, context in enumerate(message.context["data"], 1):
            print(f"\n--- 结果 {i} ---")
            print(context[:300] + "..." if len(context) > 300 else context)



查询: 中国最著名的宫殿是什么？

--- 结果 1 ---
# 故宫

## 故宫信息整理

**一、 基本信息**

*   **名称：** 故宫（又称紫禁城）
*   **身份：** 中国明清两代的皇家宫殿。
*   **别称：** 紫禁城。

**二、 地理与历史**

*   **位置：** 位于北京市中心，位于北京中轴线。
*   **建造时间：** 始建于明成祖永乐四年（1406年），永乐十八年（1420年）落成。

**三、 建筑规模与地位**

*   **规模：** 占地面积约72万平方米，建筑面积约15万平方米。
*   **结构：** 拥有七十多座大小宫殿和九千余间房屋，是世界上现存规模最大、保存最为完整的木质结构古建筑之一。
*...

--- 结果 2 ---
# 故宫博物院

# 故宫博物院

- **位置**：北京市中心，北京中轴线上  
- **别称**：紫禁城  
- **历史背景**：  
  - 始建于明成祖永乐四年（1406年）  
  - 永乐十八年（1420年）落成  
  - 为中国明清两代的皇家宫殿  

- **建筑规模**：  
  - 占地面积：72万平方米  
  - 建筑面积：约15万平方米  
  - 拥有大小宫殿七十多座，房屋九千余间  

- **特色与价值**：  
  - 世界现存规模最大、保存最完整的木质结构古建筑群之一  
  - 被誉为世界五大宫之首（北京故宫、法国凡尔赛宫、英国白金汉宫、美国白宫、俄罗...

--- 结果 3 ---
# 雍和宫

## 雍和宫信息整理

### 基本信息

*   **名称：** 雍和宫
*   **地址：** 北京市东城区雍和宫大街12号
*   **地位：** 清朝规格最高的皇家寺院，也是全国重点寺院。

### 历史沿革

*   **起源：** 原为雍正帝胤禛做贝勒和郡王的府邸。
*   **演变：**
    *   雍正三年（1725年）：改为雍亲王府。
    *   雍正十三年（1735年）：雍正帝驾崩于此，其子乾隆帝将其改为皇家寺院。

### 建筑与规模

*   **占地面积：** 66000平方米
*   **建筑面积：** 26000平方米
*   **主要建筑：...

查询: 有哪些著名的古城景区？

--- 结果 1 -